In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from pathlib import Path

from earth2studio_corrdiff_test.divergence import compute_divergence

filepath = Path.home() / "norcorrdiff_results/regression_carra2_230426/regression_output.nc"

In [ ]:
ds_truth = xr.open_dataset(filepath, group="truth")
ds_prediction = xr.open_dataset(filepath, group="prediction")
ds_input = xr.open_dataset(filepath, group="input")

print("input vars:", list(ds_input.data_vars))
print("prediction vars:", list(ds_prediction.data_vars))
print("truth vars:", list(ds_truth.data_vars))

print("input dims:", ds_input.dims)
print("prediction dims:", ds_prediction.dims)
print("truth dims:", ds_truth.dims)

print("input coords:", list(ds_input.coords))
print("prediction coords:", list(ds_prediction.coords))
print("truth coords:", list(ds_truth.coords))

In [ ]:
target_time = np.datetime64("2023-10-04T18:00:00")

time_values = ds_input["time"].values
actual_time = None

if np.issubdtype(time_values.dtype, np.datetime64):
    time_index = int(np.argmin(np.abs(time_values - target_time)))
    actual_time = np.datetime_as_string(time_values[time_index], unit="s")
else:
    datetime_candidates = ["valid_time", "datetime", "time_datetime", "time"]
    datetime_values = None
    for name in datetime_candidates:
        if name in ds_input.coords and np.issubdtype(ds_input.coords[name].dtype, np.datetime64):
            datetime_values = ds_input.coords[name].values
            break
        if name in ds_input.data_vars and np.issubdtype(ds_input[name].dtype, np.datetime64):
            datetime_values = ds_input[name].values
            break

    if datetime_values is not None and datetime_values.shape[0] == time_values.shape[0]:
        time_index = int(np.argmin(np.abs(datetime_values - target_time)))
        actual_time = np.datetime_as_string(datetime_values[time_index], unit="s")
    else:
        time_index = 0
        actual_time = str(time_values[time_index])

print(f"Requested time: {target_time}")
print(f"Using index: {time_index}")
print(f"Using nearest available time: {actual_time}")

In [ ]:
def _select_var(ds: xr.Dataset, prefix: str, component: str) -> xr.DataArray:
    preferred = [
        f"{prefix}_{component}10m",
        f"{prefix}_{component}10",
        f"{prefix}_{component}",
    ]
    for name in preferred:
        if name in ds.data_vars:
            return ds[name].isel(time=time_index)

    matches = [
        name
        for name in ds.data_vars
        if name.startswith(f"{prefix}_{component}") and "10" in name
    ]
    if not matches:
        matches = [name for name in ds.data_vars if name.startswith(f"{prefix}_{component}")]
    if not matches:
        raise KeyError(f"Could not find {component} wind component for prefix '{prefix}'.")

    return ds[matches[0]].isel(time=time_index)


def _select_lat_lon(ds: xr.Dataset, da: xr.DataArray, prefix: str) -> tuple[np.ndarray, np.ndarray]:
    lat_candidates = [f"{prefix}_lat", "lat", "latitude", "XLAT", "xlat"]
    lon_candidates = [f"{prefix}_lon", "lon", "longitude", "XLONG", "xlong"]

    lat = None
    lon = None

    for name in lat_candidates:
        if name in da.coords:
            lat = da.coords[name]
            break
        if name in ds.coords:
            lat = ds.coords[name]
            break
        if name in ds.data_vars:
            lat = ds[name].isel(time=time_index) if "time" in ds[name].dims else ds[name]
            break

    for name in lon_candidates:
        if name in da.coords:
            lon = da.coords[name]
            break
        if name in ds.coords:
            lon = ds.coords[name]
            break
        if name in ds.data_vars:
            lon = ds[name].isel(time=time_index) if "time" in ds[name].dims else ds[name]
            break

    if lat is None or lon is None:
        raise KeyError(f"Could not find lat/lon coordinates for prefix '{prefix}'.")

    lat = np.asarray(lat)
    lon = np.asarray(lon)
    data_shape = da.shape

    if lat.ndim == 1 and lon.ndim == 1:
        lon, lat = np.meshgrid(lon, lat)

    if lat.shape != data_shape:
        lat = np.broadcast_to(lat, data_shape)
    if lon.shape != data_shape:
        lon = np.broadcast_to(lon, data_shape)

    return lat, lon

In [ ]:
u_input = _select_var(ds_input, "x", "u")
v_input = _select_var(ds_input, "x", "v")
lat_input, lon_input = _select_lat_lon(ds_input, u_input, "x")
wind_input = np.sqrt(u_input.values**2 + v_input.values**2)

u_pred = _select_var(ds_prediction, "y", "u")
v_pred = _select_var(ds_prediction, "y", "v")
lat_pred, lon_pred = _select_lat_lon(ds_prediction, u_pred, "y")
wind_pred = np.sqrt(u_pred.values**2 + v_pred.values**2)

u_truth = _select_var(ds_truth, "y", "u")
v_truth = _select_var(ds_truth, "y", "v")
lat_truth, lon_truth = _select_lat_lon(ds_truth, u_truth, "y")
wind_truth = np.sqrt(u_truth.values**2 + v_truth.values**2)

In [ ]:
central_lon = float(np.nanmean(lon_truth))

projection = ccrs.LambertConformal(central_longitude=central_lon)

fig = plt.figure(figsize=(18, 6))

panels = [
    (wind_input, lat_input, lon_input, "Input wind_speed", "Greens"),
    (wind_pred, lat_pred, lon_pred, "Prediction wind_speed", "Greens"),
    (wind_truth, lat_truth, lon_truth, "Truth wind_speed", "Greens"),
]

for i, (field, lat2d, lon2d, title, cmap) in enumerate(panels, start=1):
    ax = fig.add_subplot(1, 3, i, projection=projection)
    m = ax.pcolormesh(
        lon2d,
        lat2d,
        field,
        transform=ccrs.PlateCarree(),
        cmap=cmap,
    )
    plt.colorbar(m, ax=ax, shrink=0.7, label="m s^-1")
    ax.coastlines()
    ax.gridlines()
    ax.set_title(f"{title}\n{actual_time}")

plt.tight_layout()

In [ ]:
div_input = compute_divergence(u_input.values, v_input.values, lat_input, lon_input)
div_pred = compute_divergence(u_pred.values, v_pred.values, lat_pred, lon_pred)
div_truth = compute_divergence(u_truth.values, v_truth.values, lat_truth, lon_truth)

In [ ]:
abs_max = np.nanmax(np.abs(np.stack([div_input, div_pred, div_truth])))
vmin, vmax = -abs_max, abs_max

fig = plt.figure(figsize=(18, 6))

panels = [
    (div_input, lat_input, lon_input, "Input divergence"),
    (div_pred, lat_pred, lon_pred, "Prediction divergence"),
    (div_truth, lat_truth, lon_truth, "Truth divergence"),
]

for i, (field, lat2d, lon2d, title) in enumerate(panels, start=1):
    ax = fig.add_subplot(1, 3, i, projection=projection)
    m = ax.pcolormesh(
        lon2d,
        lat2d,
        field,
        transform=ccrs.PlateCarree(),
        cmap="RdBu_r",
        vmin=vmin,
        vmax=vmax,
    )
    plt.colorbar(m, ax=ax, shrink=0.7, label="s^-1")
    ax.coastlines()
    ax.gridlines()
    ax.set_title(f"{title}\n{actual_time}")

plt.tight_layout()

In [ ]:
wind_mae_pred_vs_truth = float(np.nanmean(np.abs(wind_pred - wind_truth)))
wind_mae_input_vs_truth = float(np.nanmean(np.abs(wind_input - wind_truth)))

print(f"Wind speed MAE (prediction vs truth): {wind_mae_pred_vs_truth:.3f} m/s")
print(f"Wind speed MAE (input vs truth): {wind_mae_input_vs_truth:.3f} m/s")

In [ ]:
div_mae_pred_vs_truth = float(np.nanmean(np.abs(div_pred - div_truth)))
div_mae_input_vs_truth = float(np.nanmean(np.abs(div_input - div_truth)))

print(f"Divergence MAE (prediction vs truth): {div_mae_pred_vs_truth:.6e} s^-1")
print(f"Divergence MAE (input vs truth): {div_mae_input_vs_truth:.6e} s^-1")

In [ ]:
comparison_stats = {
    "time": actual_time,
    "wind_mae_pred_vs_truth_mps": wind_mae_pred_vs_truth,
    "wind_mae_input_vs_truth_mps": wind_mae_input_vs_truth,
    "div_mae_pred_vs_truth_s-1": div_mae_pred_vs_truth,
    "div_mae_input_vs_truth_s-1": div_mae_input_vs_truth,
    "shape_input": wind_input.shape,
    "shape_prediction": wind_pred.shape,
    "shape_truth": wind_truth.shape,
}

comparison_stats

In [ ]:
# Optional: persist arrays without assuming identical grids
np.savez(
    "outputs/check_carra2_wind_and_divergence.npz",
    wind_input=wind_input,
    wind_pred=wind_pred,
    wind_truth=wind_truth,
    div_input=div_input,
    div_pred=div_pred,
    div_truth=div_truth,
)